# 01. 데이터 구조 파악 (Data Overview)

- **작성자**: kjh
- **주제**: 제조 생산데이터 기반 전력사용량 예측 및 최대피크 위험조건 분석
- **목적**: 원본 데이터셋의 파일 구조, 인코딩, 컬럼 프로파일, 시간 간격, 전력 특성 파악

---

## 0. 환경 설정 및 모듈 로드

In [ ]:
import sys
from pathlib import Path

# 프로젝트 루트 경로를 sys.path에 추가하여 src 모듈 임포트 가능하도록 설정
ROOT_DIR = Path.cwd().resolve().parent.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

from src.config import DATA_DIR, REPORTS_DIR, FIGURES_DIR, check_data_dir
from src.viz import set_korean_font
from src.load import list_data_files, read_file_safe

# 한글 폰트 적용
set_korean_font()

# 데이터 디렉토리 유효성 점검
check_data_dir()

## A. 파일 단위 탐색 (File Overview)
- 하위 폴더 포함 전체 파일 목록, 파일 형식, 크기, 인코딩 점검
- 행/열 수 및 데이터 설명 문서 여부 확인

In [ ]:
# 1. 전체 파일 목록 조회
files_info = list_data_files(DATA_DIR)
df_files = pd.DataFrame(files_info)
print(f"총 발견된 파일 개수: {len(df_files)}개")
display(df_files)

# 2. 파일별 인코딩 및 행/열 크기 확인 (샘플 로딩)
file_summary = []
data_cache = {}  # 파일별 로드된 DataFrame 캐시

for f in files_info:
    fpath = f['path']
    ext = f['extension']
    if ext in ['.csv', '.xlsx', '.xls', '.parquet']:
        # 1000행 샘플 읽기
        sample_df, enc = read_file_safe(fpath, sample_rows=1000)
        if sample_df is not None:
            data_cache[f['filename']] = sample_df
            file_summary.append({
                '파일명': f['filename'],
                '확장자': ext,
                '크기(MB)': f['size_mb'],
                '성공 인코딩': enc,
                '샘플 행 수': len(sample_df),
                '열(컬럼) 수': len(sample_df.columns)
            })
        else:
            file_summary.append({
                '파일명': f['filename'],
                '확장자': ext,
                '크기(MB)': f['size_mb'],
                '성공 인코딩': '로드 실패',
                '샘플 행 수': None,
                '열(컬럼) 수': None
            })
    else:
        # 설명 문서(txt, pdf, hwp, docx 등) 여부 확인
        file_summary.append({
            '파일명': f['filename'],
            '확장자': ext,
            '크기(MB)': f['size_mb'],
            '성공 인코딩': '문서/기타 파일',
            '샘플 행 수': None,
            '열(컬럼) 수': None
        })

df_file_summary = pd.DataFrame(file_summary)
display(df_file_summary)

## B. 컬럼 단위 탐색 및 프로파일링 (Column Profiling)
- 컬럼명, 데이터 타입, 결측 비율, 고유값 수, 최솟값/최댓값
- 도메인 관점의 분류 (추정): 전력(타깃 후보) / 시간 / 생산량 / 제품·공정 / 설비 상태 / 환경 / 식별자 / 기타

In [ ]:
def classify_column_domain(col_name: str) -> str:
    """컬럼명을 바탕으로 도메인 역할을 '추정'으로 분류합니다."""
    col_lower = col_name.lower()
    if any(k in col_lower for k in ['power', 'kw', 'kwh', '전력', '전력량', '전력소비', '유효전력']):
        return '전력 (타깃 후보 추정)'
    elif any(k in col_lower for k in ['time', 'date', '일시', '시간', 'timestamp', 'datetime']):
        return '시간 (추정)'
    elif any(k in col_lower for k in ['count', 'prod', 'qty', '생산량', '수량', '생산']):
        return '생산량 (추정)'
    elif any(k in col_lower for k in ['recipe', 'item', 'product', '품목', '제품', '공정', 'model']):
        return '제품·공정 (추정)'
    elif any(k in col_lower for k in ['temp', 'humi', '온도', '습도', '압력', '진동', 'rpm', 'volt', 'curr']):
        return '설비 상태/센서 (추정)'
    elif any(k in col_lower for k in ['id', 'line', 'eqp', '설비', '호기', '코드', 'code']):
        return '식별자 (추정)'
    else:
        return '기타 (추정)'

col_profile_list = []
for fname, df in data_cache.items():
    for col in df.columns:
        s = df[col]
        null_ratio = s.isna().mean() * 100
        n_unique = s.nunique(dropna=True)
        min_val = s.min() if np.issubdtype(s.dtype, np.number) else (str(s.dropna().iloc[0]) if len(s.dropna()) > 0 else None)
        max_val = s.max() if np.issubdtype(s.dtype, np.number) else (str(s.dropna().iloc[-1]) if len(s.dropna()) > 0 else None)
        category = classify_column_domain(col)
        
        col_profile_list.append({
            '파일명': fname,
            '컬럼명': col,
            '자료형': str(s.dtype),
            '결측치 수': s.isna().sum(),
            '결측 비율(%)': round(null_ratio, 2),
            '고유값 수': n_unique,
            '최솟값': min_val,
            '최댓값': max_val,
            '의미 추정 분류': category
        })

df_col_profile = pd.DataFrame(col_profile_list)
display(df_col_profile)

## C. 시간 정보 점검 (Temporal Overview)
- 타임스탬프 형식 및 전체 수집 기간 파악
- 측정 주기(간격) 분포 및 누락/중복 타임스탬프 점검

In [ ]:
time_cols = df_col_profile[df_col_profile['의미 추정 분류'] == '시간 (추정)']['컬럼명'].tolist()
print(f"탐지된 시간 컬럼 후보: {time_cols}")

for fname, df in data_cache.items():
    for col in df.columns:
        if col in time_cols:
            print(f"\n[{fname}] - 시간 컬럼 '{col}' 점검")
            try:
                t_series = pd.to_datetime(df[col])
                print(f" - 시작 시간: {t_series.min()}")
                print(f" - 종료 시간: {t_series.max()}")
                print(f" - 총 기간: {t_series.max() - t_series.min()}")
                print(f" - 중복 타임스탬프 수: {t_series.duplicated().sum()}")
                
                # 시간 간격 차이 계산
                diffs = t_series.sort_values().diff().dropna()
                print(" - 측정 주기(시간 간격) 분포:")
                print(diffs.value_counts().head(5))
            except Exception as e:
                print(f" - datetime 변환 실패: {e}")

## D. 파일 간 관계 및 구조 (Relational Overview)
- 공통 식별자(ID) 키 후보 탐색
- 설비/공정/일자별 분할 구조 확인

In [ ]:
# 파일이 2개 이상일 경우 공통 컬럼 및 관계 확인
cached_names = list(data_cache.keys())
if len(cached_names) >= 2:
    print("파일 간 공통 컬럼(Join Key 후보):")
    for i in range(len(cached_names)):
        for j in range(i + 1, len(cached_names)):
            f1, f2 = cached_names[i], cached_names[j]
            common = set(data_cache[f1].columns).intersection(set(data_cache[f2].columns))
            print(f" - [{f1}] vs [{f2}] 공통 컬럼: {common}")
else:
    print("단일 데이터 파일 구성이거나 캐시된 파일이 1개입니다.")

## E. 전력 특성 점검 (Power Characteristics)
- 전력 컬럼의 단위 추정 (kWh, kW 등)
- 누적값(상승형) vs 구간값(순시형) 여부
- 음수값, 0 연속 구간, 이상치/극단값 점검

In [ ]:
power_cols = df_col_profile[df_col_profile['의미 추정 분류'].str.contains('전력')]['컬럼명'].tolist()
print(f"전력 관련 컬럼 후보: {power_cols}")

for fname, df in data_cache.items():
    for col in power_cols:
        if col in df.columns:
            s = pd.to_numeric(df[col], errors='coerce').dropna()
            print(f"\n[{fname}] - 전력 컬럼 '{col}' 특성")
            print(f" - 평균: {s.mean():.4f}, 중앙값: {s.median():.4f}, 표준편차: {s.std():.4f}")
            print(f" - 최소: {s.min():.4f}, 최대: {s.max():.4f}")
            print(f" - 음수 개수: {(s < 0).sum()}개 (비정상 값 추정)")
            print(f" - 0 값 개수: {(s == 0).sum()}개 (설비 정지 또는 대기전력 미측정 추정)")
            
            # 단조 증가(누적값) 여부 간이 점검
            is_monotonic = s.is_monotonic_increasing
            print(f" - 단조 증가(누적 전력량) 여부: {is_monotonic} (False면 순시 전력/구간 전력량 추정)")

## 4. 요약 결과 자동 저장 (Reports & Figures)
- `reports/column_summary.csv` 저장 (개인정보 및 원본 행 데이터 배제, 통계치만)
- `reports/figures/`에 결측 패턴 및 주요 분포 시각화 저장

In [ ]:
# 1. 컬럼 프로파일 요약표 저장
summary_csv_path = REPORTS_DIR / 'column_summary.csv'
df_col_profile.to_csv(summary_csv_path, index=False, encoding='utf-8-sig')
print(f"[저장 완료] 컬럼 요약 보고서: {summary_csv_path}")

# 2. 결측치 패턴 시각화 및 저장
for fname, df in data_cache.items():
    if df.isna().sum().sum() > 0:
        fig = plt.figure(figsize=(10, 4))
        msno.matrix(df)
        plt.title(f'[{fname}] 결측치 매트릭스 패턴 (추정)', fontsize=12)
        fig_name = f"missing_{fname.replace('.', '_')}.png"
        save_path = FIGURES_DIR / fig_name
        plt.savefig(save_path, bbox_inches='tight', dpi=150)
        plt.close()
        print(f"[저장 완료] 결측치 시각화: {save_path}")
    else:
        print(f"[{fname}] 결측치가 없어 결측 매트릭스 생성을 건너뜁니다.")

print("\n모든 기본 점검 및 보고서 산출물 저장이 완료되었습니다.")